In [2]:
from graph2mat4abn.tools.import_utils import load_config, get_object_from_module
from graph2mat4abn.tools.tools import get_basis_from_structures_paths, get_kwargs, load_model
from graph2mat4abn.tools.scripts_utils import get_model_dataset, init_mace_g2m_model
from graph2mat4abn.tools.script_plots import update_loss_plots, plot_grad_norms
from torch_geometric.data import DataLoader
from graph2mat4abn.tools.scripts_utils import generate_g2m_dataset_from_paths
from graph2mat4abn.tools.notebook_utils import add_dotdot_to_str
from pathlib import Path
from mace.modules import MACE, RealAgnosticResidualInteractionBlock
from graph2mat.models import MatrixMACE
from graph2mat.bindings.e3nn import E3nnGraph2Mat
import torch
import warnings
from graph2mat import BasisTableWithEdges

warnings.filterwarnings("ignore", message="The TorchScript type system doesn't support")
warnings.filterwarnings("ignore", message=".*is not a known matrix type key.*")

from joblib import dump, load
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import sisl
import pickle

def save_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f)

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

/home/ICN2/alapena/miniconda3/envs/graph2mat_upt/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  _Jd, _W3j_flat, _W3j_indic

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


In [3]:
# Load model, generate prediction and save it (so that we don't have to load the model again and again)

# Select the model to evaluate
model_dir = Path("../results/block_type_mse_nonzero_globalsquarenorm_1e-3_2-8ATOMS") 
model_name = "checkpoints/model_epoch_50000.tar"

# Structure
paths = [
    # Path('../dataset/SHARE_OUTPUTS_2_ATOMS/52b6-d4b4-4aa1-bf10-8c7d44c978d3'), 
    # Path('../dataset/SHARE_OUTPUTS_8_ATOMS/02e5-66b7-491e-a2a9-492390da1112'),
    Path('../dataset/SHARE_OUTPUTS_8_ATOMS/e0f4-41e6-4ebc-a38b-f41adc8a7e1f'),
]

# Directory to save the results
split = Path(model_name).parts[0].split("_")[0]
savedir = Path(f"../a-plots/{model_dir.parts[-1]}_{split}")

In [4]:
config = load_config(model_dir / "config.yaml")

# Basis generation (needed to initialize the model)
train_paths, val_paths = get_model_dataset(model_dir, verbose=False)

# We have to add "../" to the relative paths because we are in notebooks/
train_paths = [Path(add_dotdot_to_str(p)) for p in train_paths]
val_paths = [Path(add_dotdot_to_str(p)) for p in val_paths]

paths_basis = train_paths + val_paths
basis = get_basis_from_structures_paths(paths_basis, verbose=True, num_unique_z=config["dataset"].get("num_unique_z", None))
table = BasisTableWithEdges(basis)

print("Initializing model...")
model, optimizer, lr_scheduler, loss_fn = init_mace_g2m_model(config, table)

# Load the model
model_path = model_dir / model_name
model, checkpoint, optimizer, lr_scheduler = load_model(model, optimizer, model_path, lr_scheduler=None, initial_lr=None, device='cpu')
history = checkpoint["history"]
print(f"Loaded model in epoch {checkpoint["epoch"]} with training loss {checkpoint["train_loss"]} and validation loss {checkpoint["val_loss"]}.")

Basis computation.
Number of structures to look on: 582
Looking for unique atoms in each structure...


0it [00:00, ?it/s]

1it [00:00, 34.89it/s]

Found enough basis points. Breaking the search...
Found enough basis points. Breaking the search...
Found the following atomic numbers: [5, 7]
Corresponding path indices: [0, 0]
Basis with 2 elements built!

Basis for atom 0.
	Atom type: 5
	Basis: ((2, 0, 1), (2, 1, -1), (1, 2, 1))
	Basis convention: siesta_spherical
	R: [3.02420918 2.02341372 3.73961942 3.73961942 3.73961942 2.51253945
 2.51253945 2.51253945 3.73961942 3.73961942 3.73961942 3.73961942
 3.73961942]

Basis for atom 1.
	Atom type: 7
	Basis: ((2, 0, 1), (2, 1, -1), (1, 2, 1))
	Basis convention: siesta_spherical
	R: [2.25704422 1.4271749  2.78012609 2.78012609 2.78012609 1.75309697
 1.75309697 1.75309697 2.78012609 2.78012609 2.78012609 2.78012609
 2.78012609]
Initializing model...



/home/ICN2/alapena/miniconda3/envs/graph2mat_upt/lib/python3.12/site-packages/mace/modules/blocks.py:187: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(atomic_energies, dtype=torch.get_default_dtype()),


Using Optimizer Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
LR Scheduler:  ReduceLROnPlateau
Arguments:  None
Keyword arguments:  {'cooldown': 0, 'eps': 0.0, 'factor': 0.9, 'min_lr': 1e-09, 'mode': 'min', 'patience': 100}
LOSS FN SELECTED:  block_type_mse_nonzero_globalsquarenorm
Using Loss function <class 'graph2mat.core.data.metrics.block_type_mse_nonzero_globalsquarenorm'>
Loaded model in epoch 50000 with training loss 486.90899658203125 and validation loss 101211.9375.


In [7]:
for i, path in enumerate(paths):
    
    # Inference
    dataset, processor = generate_g2m_dataset_from_paths(config, basis, table, [path], verbose=False)
    dataloader = DataLoader(dataset, 1)
    model.eval()

    data = next(iter(dataloader))

    with torch.no_grad():
        model_predictions = model(data=data)

        h_pred = processor.matrix_from_data(data, predictions={"node_labels": model_predictions["node_labels"], "edge_labels": model_predictions["edge_labels"]})[0]
        h_true = processor.matrix_from_data(data)[0]

    # Save results
    # n_atoms = path.parent.name.split("_")[-2]
    # name = f"{n_atoms}atm_{path.name}"
    # savedir_result = savedir / name
    # savedir_result.mkdir(parents=True, exist_ok=True)

    # save_pickle(h_true, savedir_result / "h_true.pkl")
    # save_pickle(sisl.get_sile(path / "aiida.HSX").read_geometry(), savedir_result / "geometry.pkl")
    # save_pickle(h_pred, savedir_result / "h_pred.pkl")

1it [00:00,  4.24it/s]
/home/ICN2/alapena/miniconda3/envs/graph2mat_upt/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Keeping all the dataset in memory.


In [6]:
def force_zeroes_below_threshold(data, model_predictions, threshold):
    """Sets model predictions to zero where true values are under a certain threshold.

    Args:
        data (TorchBasisMatrixData or TorchBasisMatrixDataBatch): Graph2Mat data object.
        model_predictions (Dict): Dictionary containing "edge_labels" and "node_labels" as the predictions of the model.
        threshold (float): Minimum energy below to which force zero.
    """
    # Edges
    mask = torch.abs(data.edge_labels) < threshold
    new_labels = torch.where(mask, 0, model_predictions["edge_labels"])
    model_predictions["edge_labels"] = new_labels.to(data.edge_labels.device)


    # Nodes
    mask = torch.abs(data.point_labels) < threshold
    new_labels = torch.where(mask, 0, model_predictions["node_labels"])
    model_predictions["node_labels"] = new_labels.to(data.point_labels.device)


threshold = 1e-6 #eV

mask = torch.abs(data["edge_labels"]) < threshold
print(model_predictions["edge_labels"][mask]) # Before
force_zeroes_below_threshold(data, model_predictions, threshold)
print(model_predictions["edge_labels"][mask]) # After

tensor([ 0.0007,  0.0003, -0.0016,  ...,  0.0001, -0.0017,  0.0005])
tensor([0., 0., 0.,  ..., 0., 0., 0.])


# Remove edges instead

In [ ]:
import torch

data_test = dataset[0]

# Print below threshold
threshold = 1e-6 #eV
mask = torch.abs(data_test["edge_labels"]) < threshold
print(f"Threshold: {threshold:.1e} eV\n")
print("True edges below threshold:")
print(data_test["edge_labels"][mask])
print("Pred edges below threshold:")
print(model_predictions["edge_labels"][mask])

# Force zeroes
print("----------------- Force zeroes -----------------")
print("True edges below threshold:")
print(data_test["edge_labels"][mask])
print("Pred edges below threshold:")
print(model_predictions["edge_labels"][mask])

Threshold: 1.0e-06 eV
True edges below threshold:
tensor([ 9.4480e-09,  1.5293e-09,  6.0715e-07,  ..., -2.5675e-25,
        -3.6967e-22, -4.0261e-22])
Pred edges below threshold:
tensor([ 0.0007,  0.0003, -0.0016,  ...,  0.0001, -0.0017,  0.0005])


In [24]:
for i in zip(data_test.edge_index[0], data_test.edge_index[1]):
    print(i)

(tensor(7), tensor(7))
(tensor(7), tensor(7))
(tensor(7), tensor(4))
(tensor(4), tensor(7))
(tensor(7), tensor(5))
(tensor(5), tensor(7))
(tensor(7), tensor(6))
(tensor(6), tensor(7))
(tensor(7), tensor(7))
(tensor(7), tensor(7))
(tensor(7), tensor(7))
(tensor(7), tensor(7))
(tensor(4), tensor(7))
(tensor(7), tensor(4))
(tensor(5), tensor(6))
(tensor(6), tensor(5))
(tensor(7), tensor(6))
(tensor(6), tensor(7))
(tensor(7), tensor(7))
(tensor(7), tensor(7))
(tensor(7), tensor(4))
(tensor(4), tensor(7))
(tensor(7), tensor(5))
(tensor(5), tensor(7))
(tensor(7), tensor(6))
(tensor(6), tensor(7))
(tensor(7), tensor(4))
(tensor(4), tensor(7))
(tensor(7), tensor(5))
(tensor(5), tensor(7))
(tensor(7), tensor(6))
(tensor(6), tensor(7))
(tensor(4), tensor(6))
(tensor(6), tensor(4))
(tensor(4), tensor(7))
(tensor(7), tensor(4))
(tensor(4), tensor(4))
(tensor(4), tensor(4))
(tensor(7), tensor(5))
(tensor(5), tensor(7))
(tensor(7), tensor(4))
(tensor(4), tensor(7))
(tensor(7), tensor(5))
(tensor(5),

In [ ]:
# Remove edges that should be zero.
for data in dataset:

    # Get indices
    mask = torch.abs(data["edge_labels"]) < threshold
    new_labels = torch.where(mask, 0, data["edge_labels"])
    